# Практикум 2. Разведочный анализ: очистка, описание, визуализация

**Курс «Анализ больших данных»**

---

### Как работать с этим блокнотом

1. Выполняйте ячейки **по порядку**: `Shift + Enter`.
2. Подойдёт Jupyter (Anaconda или python.org + pip) или Google Colab — см. инструкцию в информационном
   разделе курса. Нужны numpy, pandas, scipy, matplotlib; sklearn и statsmodels — по желанию (есть в Colab).
3. 📘 — разбор: запустите и посмотрите результат. ✏️ — задание: допишите код вместо `...` или `# ваш код`, затем запустите
   ячейку проверки с `assert` — если ошибок нет, решение верно.
   💡 — решение под треугольником, открывайте после честной попытки.
4. Перед сдачей: **Kernel → Restart & Run All**.

### Что будет к концу блокнота

Вы проведёте аудит «сырой» выгрузки бронирований «КубаньТура» (9 видов дефектов), очистите её и
сверите с эталоном, приведёте данные к аккуратному виду, опишете цены статистиками по группам, построите графики
распределения, сравнения и тенденции и увидите, как смешение групп прячет связь признаков. Каждое задание
проверяется ячейкой с `assert`.

In [ ]:
import json, os, sys, time, random
from datetime import datetime, timedelta, timezone
from pprint import pprint
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 140)
BLUE, ORANGE = "#34519E", "#CF7A2A"      # цвета серий на графиках
NAVY, GREY = "#1E2761", "#9AA3B5"
IN_COLAB = "google.colab" in sys.modules
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| Colab" if IN_COLAB else "| локально")

Данные — кейс **«КубаньТур»** (условные; первая ячейка работает ≈ 3 с). Запустите её.

In [ ]:
# Сквозной кейс курса «Анализ больших данных» — «КубаньТур»: условная платформа бронирования
# средств размещения Краснодарского края. Все отели, туристы и бронирования вымышлены; генерация детерминирована
# (seed 2026), у всех получаются одинаковые числа. Только стандартная библиотека.
#   HOTELS          — 240 средств размещения: город, тип, звёзды, номера, персонал, расстояние до моря, удобства,
#                     рейтинг, базовая цена номера в сутки (высокий сезон)
#   bookings(n)     — ЛЕНИВЫЙ генератор n бронирований (словари) за 01.01.2025–30.09.2026
#   BOOKINGS        — 60 000 бронирований (список словарей) — основной набор для pandas
#   booking_lines(n)— ленивый генератор строк CSV «id;дата;отель;город;канал;ночей;гостей;сумма;отмена»
#   RAW_BOOKINGS    — 20 000 строк «сырой» выгрузки с дефектами (см. RAW_DEFECTS) — для разведочного анализа
import math as _m
import random as _random
from datetime import date as _date, timedelta as _td

_SEED = 2026
_TITLE = "КубаньТур"
_R = _random.Random(_SEED)
_D0, _D1 = _date(2025, 1, 1), _date(2026, 9, 30)

_CITIES = [  # город, доля отелей, у моря?, эффект цены (лог), «горный»?
    ("Сочи", 0.28, True, 0.30, False), ("Анапа", 0.17, True, 0.00, False), ("Геленджик", 0.14, True, 0.15, False),
    ("Краснодар", 0.12, False, 0.05, False), ("Красная Поляна", 0.09, False, 0.35, True),
    ("Туапсе", 0.08, True, -0.10, False), ("Ейск", 0.06, True, -0.25, False), ("Абрау-Дюрсо", 0.06, True, 0.20, False)]
_TYPES = [  # тип, доля, звёзды (мин, макс), номера (мин, макс), персонал на номер, эффект цены (лог)
    ("отель", 0.40, 2, 5, 40, 400, 0.45, 0.10), ("гостевой дом", 0.25, 0, 2, 5, 25, 0.15, -0.15),
    ("апартаменты", 0.15, 0, 3, 20, 120, 0.12, 0.05), ("санаторий", 0.10, 3, 4, 100, 500, 0.80, 0.20),
    ("хостел", 0.10, 0, 0, 10, 40, 0.10, -0.80)]
_WORDS = ["Бриз", "Волна", "Магнолия", "Кипарис", "Лазурь", "Прибой", "Пальма", "Чайка", "Маяк", "Риф", "Парус",
          "Гранат", "Лаванда", "Олива", "Горизонт", "Альпика", "Эдельвейс", "Кубань", "Таврия", "Фрегат", "Аврора",
          "Ривьера", "Жемчуг", "Солнце", "Ласточка", "Каштан", "Платан", "Янтарь", "Нептун", "Медуза"]
_PREFIX = {"отель": "Отель", "гостевой дом": "Гостевой дом", "апартаменты": "Апартаменты",
           "санаторий": "Санаторий", "хостел": "Хостел"}


def _pick(r, items, weights):
    return r.choices(items, weights=weights, k=1)[0]


HOTELS = []
for _i in range(1, 241):
    _c = _pick(_R, _CITIES, [c[1] for c in _CITIES])
    _t = _pick(_R, _TYPES, [t[1] for t in _TYPES])
    _stars = _R.randint(_t[2], _t[3])
    _rooms = _R.randint(_t[4], _t[5])
    _staff = max(1, round(_rooms * _t[6] * _R.lognormvariate(0, 0.15)))
    if _c[2]:
        _dist = round(min(4.0, _R.lognormvariate(_m.log(0.6), 0.8)), 2)
    elif _c[4]:
        _dist = round(40 + _R.uniform(-3, 3), 1)
    else:
        _dist = round(_R.uniform(115, 150), 1)
    _pool = _R.random() < 0.10 + 0.15 * _stars
    _spa = _R.random() < (0.05 + 0.12 * _stars + (0.4 if _t[0] == "санаторий" else 0))
    _breakfast = _R.random() < 0.3 + 0.12 * _stars
    _parking = _R.random() < 0.55 + (0.2 if not _c[2] else 0)
    _area = round(16 + 4.2 * _stars + _R.gauss(0, 3) + (12 if _t[0] == "апартаменты" else 0), 1)
    _rating = round(min(9.9, max(5.0, 6.4 + 0.45 * _stars - 0.25 * _m.log(_dist + 0.1) * (1 if _c[2] else 0)
                                + 0.3 * _spa + _R.gauss(0, 0.45))), 1)
    _lp = (_m.log(1600) + 0.33 * _stars + 0.22 * _pool + 0.18 * _spa + 0.10 * _breakfast + 0.012 * (_area - 25)
           - (0.12 * _m.log(_dist + 0.1) if _c[2] else 0) + _c[3] + _t[7] + 0.08 * (_rating - 8) + _R.gauss(0, 0.12))
    HOTELS.append({"hotel_id": f"H{_i:03d}", "name": f"{_PREFIX[_t[0]]} «{_R.choice(_WORDS)}»", "city": _c[0],
                   "type": _t[0], "stars": _stars, "rooms": _rooms, "staff": _staff, "dist_sea_km": _dist,
                   "pool": _pool, "spa": _spa, "breakfast": _breakfast, "parking": _parking, "room_area": _area,
                   "rating": _rating, "price_base": int(round(_m.exp(_lp), -1))})
_HOTEL = {h["hotel_id"]: h for h in HOTELS}
_CITY = {c[0]: c for c in _CITIES}
_REGIONS = [("Москва", 0.24), ("Краснодарский край", 0.18), ("Московская обл.", 0.10), ("Санкт-Петербург", 0.09),
            ("Ростовская обл.", 0.08), ("Самарская обл.", 0.05), ("Свердловская обл.", 0.05),
            ("Республика Татарстан", 0.05), ("Воронежская обл.", 0.04), ("другие", 0.12)]
_CHANNELS = [("сайт", 0.25, 1.00), ("приложение", 0.30, 0.97), ("агрегатор", 0.35, 1.06), ("офис", 0.10, 1.03)]
_AGES = [("18–24", 0.12), ("25–34", 0.30), ("35–44", 0.27), ("45–54", 0.16), ("55+", 0.15)]


def _season(month: int, city: tuple) -> float:
    """Сезонный множитель цены и спроса: море — пик в июле–августе, горы — зимой и летом, Краснодар — ровно."""
    if city[4]:
        return [1.25, 1.30, 1.05, 0.70, 0.75, 0.90, 1.05, 1.10, 0.85, 0.70, 0.75, 1.20][month - 1]
    if not city[2]:
        return [0.85, 0.85, 0.95, 1.00, 1.05, 1.05, 1.00, 1.00, 1.05, 1.05, 0.95, 1.00][month - 1]
    return [0.45, 0.45, 0.50, 0.60, 0.80, 1.05, 1.35, 1.40, 1.00, 0.65, 0.50, 0.55][month - 1]


def bookings(n: int, seed: int = _SEED):
    """Ленивый генератор n бронирований (словари). Одинаковый seed — одинаковые данные."""
    r = _random.Random(seed + 1)
    span = (_D1 - _D0).days
    hotel_w = [h["rooms"] ** 0.7 * (1 + (h["rating"] - 7) / 3) for h in HOTELS]
    for k in range(1, n + 1):
        h = r.choices(HOTELS, weights=hotel_w, k=1)[0]
        c = _CITY[h["city"]]
        while True:                                             # дата заезда с учётом сезонного спроса
            d = _D0 + _td(days=r.randrange(span + 1))
            if r.random() < _season(d.month, c) / 1.45:
                break
        ch = _pick(r, _CHANNELS, [x[1] for x in _CHANNELS])
        lead = min(240, int(r.expovariate(1 / (35 if ch[0] != "офис" else 55))))
        nights = max(1, min(21, int(r.lognormvariate(_m.log(4.2 if c[2] else (3.2 if c[4] else 1.8)), 0.55))))
        guests = r.choices([1, 2, 3, 4], weights=[18, 50, 18, 14], k=1)[0]
        if h["type"] == "хостел":
            guests = 1
        wknd = 1.08 if d.weekday() >= 4 else 1.0
        disc = 0.92 if lead > 60 else 1.0
        price = h["price_base"] * _season(d.month, c) * wknd * disc * ch[2] * r.lognormvariate(0, 0.08)
        price *= 1 + 0.25 * (guests - 2) * (h["type"] in ("апартаменты", "санаторий"))
        price = int(round(max(500, price), -1))
        p_cancel = 0.06 + 0.07 * (ch[0] == "агрегатор") + 0.0006 * lead
        cancelled = r.random() < p_cancel
        score = None
        if not cancelled and r.random() < 0.35:
            score = max(1, min(10, round(h["rating"] + r.gauss(0, 1.1) - (0.4 if guests >= 3 else 0))))
        yield {"booking_id": f"B{k:07d}", "created": (d - _td(days=lead)).isoformat(), "checkin": d.isoformat(),
               "hotel_id": h["hotel_id"], "city": h["city"], "type": h["type"], "channel": ch[0],
               "region": _pick(r, [x[0] for x in _REGIONS], [x[1] for x in _REGIONS]),
               "age_group": _pick(r, [x[0] for x in _AGES], [x[1] for x in _AGES]),
               "lead_days": lead, "nights": nights, "guests": guests, "price_night": price,
               "amount": price * nights, "cancelled": cancelled, "review_score": score}


BOOKINGS = list(bookings(60_000))


def booking_lines(n: int, seed: int = _SEED):
    """Ленивый генератор строк журнала «id;дата_заезда;отель;город;канал;ночей;гостей;сумма;отмена» (без заголовка)."""
    for b in bookings(n, seed):
        yield (f"{b['booking_id']};{b['checkin']};{b['hotel_id']};{b['city']};{b['channel']};{b['nights']};"
               f"{b['guests']};{b['amount']};{int(b['cancelled'])}")


# «сырая» выгрузка: те же поля, но с типичными дефектами реальных данных
_RR = _random.Random(_SEED + 7)
RAW_BOOKINGS, RAW_DEFECTS = [], {"формат даты дд.мм.гггг": 0, "город с ошибкой написания": 0,
                                 "сумма строкой «… руб.»": 0, "пропуск ночей": 0, "пропуск канала": 0,
                                 "выброс цены (×100)": 0, "выброс ночей (365)": 0, "отрицательные гости": 0,
                                 "полный дубль": 0}
for _b in BOOKINGS[:20_000]:
    _row = {k: _b[k] for k in ("booking_id", "checkin", "hotel_id", "city", "channel", "nights", "guests",
                               "price_night", "cancelled")}
    _u = _RR.random()
    if _u < 0.06:
        _row["checkin"] = _date.fromisoformat(_row["checkin"]).strftime("%d.%m.%Y"); RAW_DEFECTS["формат даты дд.мм.гггг"] += 1
    elif _u < 0.10:
        _row["city"] = _RR.choice([_row["city"].lower(), _row["city"] + " ", "г. " + _row["city"]]); RAW_DEFECTS["город с ошибкой написания"] += 1
    elif _u < 0.13:
        _row["price_night"] = f"{_row['price_night']:,} руб.".replace(",", " "); RAW_DEFECTS["сумма строкой «… руб.»"] += 1
    elif _u < 0.155:
        _row["nights"] = None; RAW_DEFECTS["пропуск ночей"] += 1
    elif _u < 0.175:
        _row["channel"] = None; RAW_DEFECTS["пропуск канала"] += 1
    elif _u < 0.180:
        _row["price_night"] = _row["price_night"] * 100; RAW_DEFECTS["выброс цены (×100)"] += 1
    elif _u < 0.184:
        _row["nights"] = 365; RAW_DEFECTS["выброс ночей (365)"] += 1
    elif _u < 0.187:
        _row["guests"] = -_row["guests"]; RAW_DEFECTS["отрицательные гости"] += 1
    RAW_BOOKINGS.append(_row)
    if _RR.random() < 0.015:
        RAW_BOOKINGS.append(dict(_row)); RAW_DEFECTS["полный дубль"] += 1

print(f"КубаньТур: отелей {len(HOTELS)}, бронирований {len(BOOKINGS)}, строк сырой выгрузки {len(RAW_BOOKINGS)}")

## 1. Аудит данных 📘

Типы, пропуски, уникальные значения, `describe` — первые признаки дефектов.

In [ ]:
# Аудит «сырой» выгрузки
raw = pd.DataFrame(RAW_BOOKINGS)
print(raw.shape)
print(raw.dtypes.astype(str).to_dict())
print("пропуски:", raw.isna().sum()[lambda s: s > 0].to_dict())
print("городов:", raw.city.nunique(), sorted(raw.city.unique())[:6])
num = pd.to_numeric(raw.price_night, errors="coerce")
print("цены-строки:", raw.price_night[num.isna()].head(3).tolist())
raw[["nights", "guests"]].apply(pd.to_numeric, errors="coerce").describe().round(1)

## 2. Очистка 📘

Даты двух форматов — явным разбором; суммы — из строк; города — по справочнику; пропуски — медианой по группе с флагом; выбросы — на логарифмической шкале; дубли.

In [ ]:
# Очистка: даты, суммы, города, пропуски, выбросы, дубли
clean = raw.copy()
iso = clean.checkin.str.match(r"\d{4}-\d{2}-\d{2}$")
clean["checkin"] = pd.to_datetime(clean.checkin.where(iso), format="%Y-%m-%d").fillna(
    pd.to_datetime(clean.checkin.where(~iso), format="%d.%m.%Y"))
price_str = clean.price_night.astype(str).str.replace("руб.", "", regex=False).str.replace(r"\s", "", regex=True)
clean["price_night"] = pd.to_numeric(price_str, errors="coerce")
canon = {c.lower(): c for c in pd.DataFrame(HOTELS).city.unique()}
clean["city"] = clean.city.str.strip().str.replace(r"^г\.\s*", "", regex=True).str.lower().map(canon)
clean["nights_missing"] = clean.nights.isna()
clean["nights"] = clean.nights.fillna(clean.groupby("city")["nights"].transform("median"))
clean["channel"] = clean.channel.fillna("неизвестно")
lp = np.log(clean.price_night)
q1, q3 = lp.quantile([0.25, 0.75])
out = (lp < q1 - 3 * (q3 - q1)) | (lp > q3 + 3 * (q3 - q1)) | (clean.nights > 30)
clean = clean[~out].copy()
clean["guests"] = clean.guests.abs()
clean = clean.drop_duplicates().drop_duplicates(subset="booking_id")
print(len(raw), "->", len(clean), "| удалено выбросов:", int(out.sum()))
print(pd.Series(RAW_DEFECTS).to_string())

## 3. Сверка с эталоном 📘

У кейса есть чистая версия тех же бронирований.

In [ ]:
# Сверка с эталоном: чистые бронирования кейса
ref = pd.DataFrame(BOOKINGS[:20_000]).set_index("booking_id")
m = clean.set_index("booking_id").join(ref[["price_night", "city", "nights"]], rsuffix="_ref", how="inner")
print(len(m), (m.price_night == m.price_night_ref).mean().round(4), (m.city == m.city_ref).mean(),
      (m.nights == m.nights_ref).mean().round(4))

## 4. Аккуратные данные и поднаборы 📘

`pivot_table` ↔ `melt`; `query`, `isin`, `between`.

In [ ]:
# Аккуратные данные и поднаборы
b = pd.DataFrame(BOOKINGS)
b["month"] = pd.to_datetime(b.checkin).dt.month
wide = b.pivot_table(index="city", columns="month", values="price_night", aggfunc="median")
long = wide.reset_index().melt(id_vars="city", var_name="month", value_name="median_price")
print(wide.shape, "->", long.shape)
summer_sea = b.query("month in [7, 8] and city in ['Сочи', 'Анапа', 'Геленджик'] and not cancelled")
print(len(summer_sea), summer_sea.price_night.median())

## 5. Описательная статистика 📘

Центр, разброс, форма; устойчивые статистики; по группам.

In [ ]:
# Описательная статистика: центр, разброс, форма; по группам
from scipy import stats
p = b.price_night
print(pd.Series({"среднее": p.mean(), "медиана": p.median(), "усечённое 5 %": stats.trim_mean(p, 0.05),
                 "ст. откл.": p.std(), "IQR": p.quantile(.75) - p.quantile(.25), "асимметрия": p.skew(),
                 "асимметрия логарифма": np.log(p).skew()}).round(2).to_string())
b.groupby("type")["price_night"].agg(["count", "median", "mean", "std"]).round(0).sort_values("median")

## 6. Визуализация 📘

Гистограмма на логарифмической шкале, ящики с усами, тенденция по месяцам.

In [ ]:
# Визуализация: распределение, группы, тенденция
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
ax[0].hist(np.log10(p), bins=60, color=BLUE); ax[0].set_title("lg(цена за ночь)")
order = b.groupby("type").price_night.median().sort_values().index
ax[1].boxplot([b[b.type == t].price_night for t in order], tick_labels=list(order), orientation="horizontal",
              flierprops=dict(marker=".", markersize=2, alpha=.3))
ax[1].set_xscale("log"); ax[1].set_title("цена по типам")
mon = b[~b.cancelled].set_index(pd.to_datetime(b[~b.cancelled].checkin)).resample("MS").price_night.median()
ax[2].plot(mon.index, mon.values, color=ORANGE); ax[2].plot(mon.index, mon.rolling(3, center=True).mean(), color=NAVY)
ax[2].set_title("медиана по месяцам"); ax[2].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

## 7. Корреляции и смешение групп 📘

Пирсон и Спирмен; связь цены с расстоянием до моря проявляется только после учёта звёздности.

In [ ]:
# Корреляции: Пирсон и Спирмен; смешение групп
h = pd.DataFrame(HOTELS)
cols = ["stars", "rating", "room_area", "rooms", "staff", "dist_sea_km", "price_base"]
print(h[cols].corr().round(2)["price_base"].to_dict())
print(h[cols].corr(method="spearman").round(2)["price_base"].to_dict())
sea = h[h.dist_sea_km < 10]
lp_, ld_ = np.log(sea.price_base), np.log(sea.dist_sea_km + 0.1)
X = np.column_stack([np.ones(len(sea)), sea.stars])
resid = lp_ - X @ np.linalg.lstsq(X, lp_, rcond=None)[0]
print("до моря и цена:", round(np.corrcoef(ld_, lp_)[0, 1], 3), "| после учёта звёзд:", round(np.corrcoef(ld_, resid)[0, 1], 3))

---

# Задания ✏️

Пишите код вместо `...` / `# ваш код`, затем запускайте **ячейку проверки** под заданием.

## Задание 1. Функция аудита

Таблица типов, пропусков, уникальных значений.

In [ ]:
# Задание 1. Функция audit(df) возвращает таблицу (DataFrame) с индексом — названиями столбцов df и колонками
# "тип" (строка dtype), "пропусков", "уникальных", "пример" (первое непустое значение). Примените к raw.
def audit(df: pd.DataFrame) -> pd.DataFrame:
    ...

In [ ]:
a = audit(raw)
assert list(a.columns) == ["тип", "пропусков", "уникальных", "пример"] and list(a.index) == list(raw.columns)
assert a.loc["nights", "пропусков"] == 488 and a.loc["channel", "пропусков"] == 361
assert a.loc["city", "уникальных"] > 8
print("Задание 1: верно")

## Задание 2. Даты и суммы

Явный разбор двух форматов дат; суммы из строк.

In [ ]:
# Задание 2. Сколько дат в raw записано НЕ в формате ГГГГ-ММ-ДД (non_iso)? Разберите все даты в checkin_dt
# (Series datetime64) явным разбором двух форматов и цены в price_num (числа). Пропусков быть не должно.
# ваш код

In [ ]:
assert non_iso == 1213
assert checkin_dt.isna().sum() == 0 and price_num.isna().sum() == 0
assert checkin_dt.min() >= pd.Timestamp("2025-01-01") and checkin_dt.max() <= pd.Timestamp("2026-09-30")
print("Задание 2: верно")

## Задание 3. Справочник городов

Нормализация написания.

In [ ]:
# Задание 3. Приведите города raw к справочнику (8 названий из HOTELS): city_ok — Series той же длины, без пропусков.
# Сколько строк было исправлено (fixed — значение изменилось)?
# ваш код

In [ ]:
assert city_ok.nunique() == 8 and city_ok.isna().sum() == 0
assert fixed == int((~raw.city.isin(pd.DataFrame(HOTELS).city.unique())).sum())
assert fixed >= RAW_DEFECTS["город с ошибкой написания"]   # больше на строки-дубли с испорченным городом
print("Задание 3: верно")

## Задание 4. Пропуски с флагом

Медиана по группе; проверка случайности пропусков.

In [ ]:
# Задание 4. В копии raw (таблица r4) заполните пропуски ночей медианой по городу (city_ok), добавьте флаг
# nights_filled, пропуски канала замените на "неизвестно". Проверьте: какая доля заполненных строк в каждом канале
# (miss_by_channel) — случайны ли пропуски ночей?
# ваш код

In [ ]:
assert r4.nights.isna().sum() == 0 and r4.channel.isna().sum() == 0
assert r4.nights_filled.sum() == raw.nights.isna().sum() >= RAW_DEFECTS["пропуск ночей"]
assert (r4.loc[r4.nights_filled, "nights"] == r4.loc[r4.nights_filled, "city"].map(raw.assign(city=city_ok).groupby("city").nights.median())).all()
print("Задание 4: верно")

## Задание 5. Робастные выбросы

z-показатель по медиане и MAD на логарифме.

In [ ]:
# Задание 5. Найдите выбросы цены робастным z-показателем на логарифмической шкале: z = |ln p − медиана| / (1,4826·MAD),
# выброс при z > 3,5 (p = price_num из задания 2). Маска price_out (bool Series). Сколько найдено и сколько из них —
# настоящие ошибки «×100» (сравните с BOOKINGS[:20 000] по booking_id: цена в 100 раз больше эталонной)?
# ваш код

In [ ]:
assert price_out.dtype == bool and len(price_out) == len(raw)
assert (price_out & true_x100).sum() >= 0.8 * true_x100.sum(), "правило должно ловить большинство ошибок ×100"
print("Задание 5: верно")

## Задание 6. Статистики по городам

Медиана, IQR, коэффициент вариации.

In [ ]:
# Задание 6. По чистым бронированиям b: для каждого города медиана, IQR и коэффициент вариации (%) цены за ночь —
# таблица city_stats (индекс — город, колонки "медиана", "IQR", "CV, %"), по убыванию медианы.
# ваш код

In [ ]:
assert city_stats["медиана"].to_dict() == {'Абрау-Дюрсо': 9000.0, 'Анапа': 9700.0, 'Геленджик': 9310.0, 'Ейск': 5540.0, 'Красная Поляна': 10300.0, 'Краснодар': 8610.0, 'Сочи': 11800.0, 'Туапсе': 5970.0}
assert abs(city_stats.loc["Сочи", "IQR"] - 14740.0) < 1e-6
assert list(city_stats["медиана"]) == sorted(city_stats["медиана"], reverse=True)
print("Задание 6: верно")

## Задание 7. Связь после учёта факторов

Остатки МНК и корреляция.

In [ ]:
# Задание 7. Для морских отелей (dist_sea_km < 10) посчитайте корреляцию ln(расстояние + 0,1) с остатками ln(цены)
# после учёта звёздности И бассейна (МНК по столбцам 1, stars, pool) — r_resid. Постройте диаграмму рассеяния
# остатков от расстояния (логарифмическая ось X).
# ваш код

In [ ]:
assert -0.6 < r_resid < -0.1, "после учёта звёзд и бассейна связь должна быть отрицательной"
assert round(float(np.corrcoef(ld_, resid)[0, 1]), 3) == -0.254
print("Задание 7: верно")

## Задание 8 ★. Мини-исследование: правила очистки

Таблица «до/после».

In [ ]:
# Задание 8 ★ (мини-исследование). Сравните правила обработки выбросов цены price_num: без очистки; три сигмы; IQR
# (k = 1,5) в рублях; IQR (k = 3) на логарифме; робастный z ≤ 3,5 на логарифме. Для каждого — сколько цен удалено,
# среднее, медиана, ст. отклонение; последняя строка — эталон (цены BOOKINGS[:20 000]). Таблица + вывод 5–7 предложений:
# какое правило ближе к эталону и почему IQR в рублях «режет» настоящие дорогие отели.
# ваш код

In [ ]:
err = (res.loc[res.index != "эталон", "среднее"] - res.loc["эталон", "среднее"]).abs()
assert err.idxmin() != "без очистки"
print("Задание 8: таблица построена; вывод проверит преподаватель")

---

## Что вы теперь умеете

| Задача | Инструмент |
|---|---|
| аудит | `dtypes`, `isna`, `nunique`, `describe` |
| очистка | `to_datetime(format=)`, `to_numeric`, справочник, `fillna` + флаг, `drop_duplicates` |
| выбросы | IQR, робастный z, логарифмическая шкала |
| описание | медиана, IQR, CV, асимметрия; `groupby().agg` |
| графики | гистограмма, ящики с усами, временной ряд |
| связь | Пирсон, Спирмен, остатки после учёта фактора |

**Дальше — тема 3 «Проверка статистических гипотез».** Различия цен по каналам и городам проверим
критериями Стьюдента, Манна–Уитни, дисперсионным анализом.